⚠️ Run all cells top-to-bottom before executing explainability steps.

In [ ]:
import pandas as pd

# Load dataset
df = pd.read_csv("../data/creditcard.csv")

# Quick look
df.head()

In [ ]:
df.columns

In [ ]:
import pandas as pd

# Load dataset
df = pd.read_csv("../data/creditcard.csv")

# Quick look
df.head()

In [ ]:
df.columns

In [ ]:
df["Class"].value_counts()

In [ ]:
df.info()

In [ ]:
df["Class"].value_counts()

In [ ]:
import matplotlib.pyplot as plt

df["Class"].value_counts().plot(kind="bar")
plt.title("Fraud vs Non-Fraud Transactions")
plt.xlabel("Class (0 = Normal,1 = Fraud)")
plt.ylabel("count")
plt.show()

In [ ]:
df["Amount"].describe()

In [ ]:
df.groupby("Class")["Amount"].describe()

In [ ]:
import seaborn as sns

plt.figure(figsize=(10,5))
sns.hisplot(df[df["Class"] == 0]["Amount"], bins=50, log_Scale=True)
plt.title("Transaction Amounts (Non-Fraud)")
plt.show()

plt.figure(figsize=(10,5))
sns.histplot(df[df["Class"] == 1]["Amount"], bins=50, log_scale=True)
plt.title("Transaction Amounts (Fraud)")
plt.show()

In [ ]:
import seaborn as sns

plt.figure(figsize=(10,5))
sns.histplot(df[df["Class"] == 0]["Amount"], bins=50, log_scale=True)
plt.title("Transaction Amounts (Non-Fraud)")
plt.show()

plt.figure(figsize=(10,5))
sns.histplot(df[df["Class"] == 1]["Amount"], bins=50, log_scale=True)
plt.title("Transaction Amounts (Fraud)")
plt.show()

In [ ]:
import seaborn as sns

plt.figure(figsize=(10,5))
sns.histplot(df[df["Class"] == 0]["Amount"], bins=50, log_scale=True)
plt.title("Transaction Amounts (Non-Fraud)")
plt.show()

plt.figure(figsize=(10,5))
sns.histplot(df[df["Class"] == 1]["Amount"], bins=50, log_scale=True)
plt.title("Transaction Amounts (Fraud)")
plt.show()

In [ ]:
import numpy as np

plt.figure(figsize=(10,5))
sns.histplot(np.log10(df[df["Class"] == 0]["Amount"] + 0.01), bins=50)
plt.title("Transaction Amounts (Non-Fraud, Log-Scaled)")
plt.show()

plt.figure(figsize=(10,5))
sns.histplot(np.log10(df[df["Class"] == 1]["Amount"] + 0.01), bins=50)
plt.title("Transaction Amounts (Fraud, Log-Scaled)")
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
sns.histplot(df["Time"], bins=50)
plt.title("Transaction Time Distribution")
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
sns.histplot(df[df["Class"] == 1]["Time"], bins=50)
plt.title("Fraud Transactions Over Time")
plt.show()

In [ ]:
corr = df.corr()
corr["Class"].sort_values(ascending=False).head(10)

Dataset is highly imbalanced
Fraud transactions show deifferent amount distributions

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["Class"])
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, X_test.shape

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit only on training data
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[["Time", "Amount"]] = scaler.fit_transform(X_train[["Time", "Amount"]])
X_test_scaled[["Time", "Amount"]] = scaler.transform(X_test[["Time", "Amount"]])

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train_scaled, y_train)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = model.predict(X_test_scaled)

print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, digits=4))

In [ ]:
from sklearn.metrics import average_precision_score, precision_recall_curve
import matplotlib.pyplot as plt

y_scores = model.predict_proba(X_test_scaled)[:, 1]

ap = average_precision_score(y_test, y_scores)
print("Average Precision (PR-AUC):", ap)

precision, recall, thresholds = precision_recall_curve(y_test, y_scores)

plt.figure()
plt.plot(recall, precision)
plt.title("Precision-Recall Curve (Baseline Logistic Regression)")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.show()

The precision–recall curve shows a strong tradeoff between fraud detection coverage and false positives. The model maintains high precision (>80%) up to ~70–80% recall, indicating strong ranking performance. This makes it suitable for explainability analysis and threshold-based decisioning.

In [ ]:
import numpy as np
from sklearn.metrics import precision_score, recall_score

threshold = 0.2  # try 0.2, 0.1, 0.05 later
y_pred_thresh = (y_scores >= threshold).astype(int)

print("Threshold:", threshold)
print("Precision:", precision_score(y_test, y_pred_thresh))
print("Recall:", recall_score(y_test, y_pred_thresh))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_thresh))

Baseline model performance

Precision/recall tradeoffs

What happens when threshold changes

In [ ]:
import shap

In [ ]:
explainer = shap.LinearExplainer(
    model,
    X_train_scaled,
    feature_names=X_train_scaled.columns
)

In [ ]:
type(model)

In [ ]:
explainer = shap.LinearExplainer(
    model,
    X_train_scaled,
    feature_names=X_train_scaled.columns
)

In [ ]:
import shap

In [ ]:
explainer = shap.LinearExplainer(
    model,
    X_train_scaled,
    feature_names=X_train_scaled.columns
)

In [ ]:
shap_values = explainer(X_test_scaled)

In [ ]:
shap.summary_plot(shap_values, X_test_scaled)

The SHAP summary plot shows that the model primarily relies on a small set of features, including V1, V14, V4, transaction amount, and V10, to detect fraud. High transaction amounts consistently push predictions toward fraud, while specific low values in anonymized PCA features strongly indicate fraudulent behavior. Less influential features such as transaction time have minimal impact, confirming insights observed during exploratory data analysis. Overall, the explanations demonstrate that the model’s decisions are driven by stable, interpretable signals rather than noise.

In [ ]:
import numpy as np

# Indices where model predicts fraud
fraud_indices = np.where(y_pred == 1)[0]

# Pick one example
idx = fraud_indices[0]
idx

In [ ]:
shap.plots.waterfall(shap_values[idx])

In [ ]:
feature_contributions = list(
    zip(
        X_test_scaled.columns,
        shap_values[idx].values
    )
)

# Sort by absolute impact
feature_contributions = sorted(
    feature_contributions,
    key=lambda x: abs(x[1]),
    reverse=True
)

feature_contributions[:5]

This transaction was flagged as fraud primarily due to unusually high values in features V14 and V10, which historically correlate with fraudulent behavior. The transaction amount also contributed moderately to the fraud prediction. These signals outweighed normal behavioral indicators, resulting in a high-risk classification.

In [ ]:
import numpy as np

# Indices where model predicts Non-Fraud
non_fraud_indices = np.where(y_pred == 0)[0]

#pick one example
idx = non_fraud_indices[0]
idx

In [ ]:
shap.plots.waterfall(shap_values[idx])

This transaction was classified as non-fraud with high confidence. While a small number of features contributed weakly toward fraud, the majority of signals—including V7, V4, V10, and a large group of other features—strongly indicated normal behavior. The combined effect of these features pushed the model output well below the baseline fraud risk, resulting in a confident non-fraud prediction.

In [ ]:
print("df:", df.shape)
print("model:", type(model))
print("X_test_scaled:", X_test_scaled.shape)
print("shap_values:", shap_values.shape)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

import shap


In [2]:
df = pd.read_csv("../data/creditcard.csv")

In [3]:
df.shape


(284807, 31)

In [4]:
X = df.drop(columns=["Class"])
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [5]:
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[["Time", "Amount"]] = scaler.fit_transform(
    X_train[["Time", "Amount"]]
)

X_test_scaled[["Time", "Amount"]] = scaler.transform(
    X_test[["Time", "Amount"]]
)

In [6]:
model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train_scaled, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

In [7]:
y_pred = model.predict(X_test_scaled)
y_scores = model.predict_proba(X_test_scaled)[:, 1]

In [8]:
explainer = shap.LinearExplainer(
    model,
    X_train_scaled,
    feature_names=X_train_scaled.columns
)

shap_values = explainer(X_test_scaled)

In [9]:
print("df:", df.shape)
print("model:", type(model))
print("X_test_scaled:", X_test_scaled.shape)
print("shap_values:", shap_values.shape)

df: (284807, 31)
model: <class 'sklearn.linear_model._logistic.LogisticRegression'>
X_test_scaled: (56962, 30)
shap_values: (56962, 30)


In [10]:
import numpy as np
import pandas as pd

def logodds_to_prob(log_odds: float) -> float:
    # logistic function
    return 1 / (1 + np.exp(-log_odds))

def get_case_evidence(case_idx: int, top_n: int = 8) -> dict:
    """
    Returns a structured evidence packet for one transaction:
    - prediction, probability
    - baseline output, final output (log-odds)
    - top SHAP drivers (feature, value, shap_value, direction)
    """
    # Model outputs
    prob = float(model.predict_proba(X_test_scaled.iloc[[case_idx]])[:, 1][0])
    pred = int(prob >= 0.5)  # default threshold; we will also support custom thresholds later
    
    # SHAP outputs for this row
    sv = shap_values[case_idx]
    base_logodds = float(sv.base_values)
    final_logodds = float(base_logodds + sv.values.sum())  # should match model output in log-odds space
    base_prob = float(logodds_to_prob(base_logodds))
    final_prob_from_shap = float(logodds_to_prob(final_logodds))
    
    # Build feature contribution table
    contrib = pd.DataFrame({
        "feature": X_test_scaled.columns,
        "feature_value_scaled": X_test_scaled.iloc[case_idx].values,
        "shap_value": sv.values
    })
    contrib["abs_shap"] = contrib["shap_value"].abs()
    contrib = contrib.sort_values("abs_shap", ascending=False).drop(columns=["abs_shap"]).head(top_n)
    
    # Direction text
    contrib["direction"] = np.where(contrib["shap_value"] >= 0, "pushes toward FRAUD", "pushes toward NON-FRAUD")
    
    evidence = {
        "case_idx": int(case_idx),
        "model_prediction_default_threshold": int(pred),
        "model_probability_fraud": prob,
        "shap_baseline_logodds": base_logodds,
        "shap_final_logodds": final_logodds,
        "shap_baseline_probability": base_prob,
        "shap_final_probability": final_prob_from_shap,
        "top_drivers": contrib.to_dict(orient="records"),
        "notes": {
            "interpretation": "Positive SHAP pushes toward fraud, negative SHAP pushes toward non-fraud. For logistic regression, SHAP is in log-odds space."
        }
    }
    return evidence

In [11]:
def generate_rule_based_explanation(evidence: dict, threshold: float = 0.5) -> str:
    prob = evidence["model_probability_fraud"]
    pred = int(prob >= threshold)
    label = "FRAUD" if pred == 1 else "NON-FRAUD"
    
    drivers = evidence["top_drivers"]
    fraud_push = [d for d in drivers if d["shap_value"] >= 0]
    nonfraud_push = [d for d in drivers if d["shap_value"] < 0]
    
    # Build bullet points (top 3 each side)
    def fmt_driver(d):
        f = d["feature"]
        v = d["feature_value_scaled"]
        s = d["shap_value"]
        direction = "↑ fraud risk" if s >= 0 else "↓ fraud risk"
        return f"- **{f}** (value={v:.3f}) contributed **{s:+.3f}** ({direction})"
    
    top_fraud = "\n".join(fmt_driver(d) for d in fraud_push[:3]) if fraud_push else "- None of the top drivers pushed toward fraud."
    top_nonfraud = "\n".join(fmt_driver(d) for d in nonfraud_push[:3]) if nonfraud_push else "- None of the top drivers pushed toward non-fraud."
    
    explanation = f"""
### Case {evidence['case_idx']} — Decision Explanation

**Model decision (threshold={threshold:.2f}):** {label}  
**Predicted fraud probability:** {prob:.4f}

#### Key drivers pushing toward FRAUD
{top_fraud}

#### Key drivers pushing toward NON-FRAUD
{top_nonfraud}

#### What this means
The model starts from a baseline risk level and adjusts the final risk using feature contributions.  
Positive SHAP contributions raise fraud risk; negative contributions reduce it. The final probability reflects the combined effect of all features.
""".strip()
    
    return explanation

In [12]:
# Choose high-confidence cases for clean reports
fraud_idx = int(np.argmax(y_scores))     # highest predicted fraud probability
nonfraud_idx = int(np.argmin(y_scores))  # lowest predicted fraud probability

fraud_idx, nonfraud_idx

(1146, 56510)

In [13]:
fraud_evidence = get_case_evidence(fraud_idx, top_n=10)
nonfraud_evidence = get_case_evidence(nonfraud_idx, top_n=10)

print(generate_rule_based_explanation(fraud_evidence, threshold=0.5))
print("\n" + "-"*80 + "\n")
print(generate_rule_based_explanation(nonfraud_evidence, threshold=0.5))

### Case 1146 — Decision Explanation

**Model decision (threshold=0.50):** FRAUD  
**Predicted fraud probability:** 1.0000

#### Key drivers pushing toward FRAUD
- **V8** (value=-37.353) contributed **+18.038** (↑ fraud risk)
- **V14** (value=-9.073) contributed **+14.420** (↑ fraud risk)
- **V7** (value=-18.751) contributed **+12.435** (↑ fraud risk)

#### Key drivers pushing toward NON-FRAUD
- **V1** (value=-13.193) contributed **-11.313** (↓ fraud risk)
- **V22** (value=-8.887) contributed **-7.332** (↓ fraud risk)

#### What this means
The model starts from a baseline risk level and adjusts the final risk using feature contributions.  
Positive SHAP contributions raise fraud risk; negative contributions reduce it. The final probability reflects the combined effect of all features.

--------------------------------------------------------------------------------

### Case 56510 — Decision Explanation

**Model decision (threshold=0.50):** NON-FRAUD  
**Predicted fraud probability:** 

In [14]:
for t in [0.5, 0.2, 0.1]:
    print(f"\n\n=== Threshold {t} ===")
    print(generate_rule_based_explanation(fraud_evidence, threshold=t))



=== Threshold 0.5 ===
### Case 1146 — Decision Explanation

**Model decision (threshold=0.50):** FRAUD  
**Predicted fraud probability:** 1.0000

#### Key drivers pushing toward FRAUD
- **V8** (value=-37.353) contributed **+18.038** (↑ fraud risk)
- **V14** (value=-9.073) contributed **+14.420** (↑ fraud risk)
- **V7** (value=-18.751) contributed **+12.435** (↑ fraud risk)

#### Key drivers pushing toward NON-FRAUD
- **V1** (value=-13.193) contributed **-11.313** (↓ fraud risk)
- **V22** (value=-8.887) contributed **-7.332** (↓ fraud risk)

#### What this means
The model starts from a baseline risk level and adjusts the final risk using feature contributions.  
Positive SHAP contributions raise fraud risk; negative contributions reduce it. The final probability reflects the combined effect of all features.


=== Threshold 0.2 ===
### Case 1146 — Decision Explanation

**Model decision (threshold=0.20):** FRAUD  
**Predicted fraud probability:** 1.0000

#### Key drivers pushing toward 

In [15]:
from pathlib import Path

Path("../outputs").mkdir(exist_ok=True)

fraud_report = generate_rule_based_explanation(fraud_evidence, threshold=0.2)
nonfraud_report = generate_rule_based_explanation(nonfraud_evidence, threshold=0.2)

Path("../outputs/fraud_case_report.md").write_text(fraud_report, encoding="utf-8")
Path("../outputs/nonfraud_case_report.md").write_text(nonfraud_report, encoding="utf-8")

"Saved: outputs/fraud_case_report.md and outputs/nonfraud_case_report.md"


'Saved: outputs/fraud_case_report.md and outputs/nonfraud_case_report.md'

In [16]:
import numpy as np

# Pick a case with predicted probability closest to 0.2
target = 0.2
border_idx = int(np.argmin(np.abs(y_scores - target)))

border_idx, y_scores[border_idx]

(31472, 0.19999001610751221)

In [17]:
border_evidence = get_case_evidence(border_idx, top_n=10)

for t in [0.5, 0.2, 0.1]:
    print("\n\n=== Threshold", t, "===")
    print(generate_rule_based_explanation(border_evidence, threshold=t))



=== Threshold 0.5 ===
### Case 31472 — Decision Explanation

**Model decision (threshold=0.50):** NON-FRAUD  
**Predicted fraud probability:** 0.2000

#### Key drivers pushing toward FRAUD
- **V12** (value=-2.935) contributed **+3.244** (↑ fraud risk)
- **V4** (value=1.421) contributed **+1.574** (↑ fraud risk)
- **V3** (value=1.898) contributed **+0.685** (↑ fraud risk)

#### Key drivers pushing toward NON-FRAUD
- **V14** (value=1.113) contributed **-1.445** (↓ fraud risk)
- **V9** (value=1.737) contributed **-0.879** (↓ fraud risk)
- **V17** (value=0.728) contributed **-0.874** (↓ fraud risk)

#### What this means
The model starts from a baseline risk level and adjusts the final risk using feature contributions.  
Positive SHAP contributions raise fraud risk; negative contributions reduce it. The final probability reflects the combined effect of all features.


=== Threshold 0.2 ===
### Case 31472 — Decision Explanation

**Model decision (threshold=0.20):** NON-FRAUD  
**Predicted 